# SDG interaction matrix (matrix A)

Rebuilds the goal-level 17x17 SDG interaction matrix from the UN Global SDG
Indicators Database, following Pradhan et al. (2017) with the
direction-of-progress and Boolean-exclusion refinements of Warchold et al.
(2022). Produces the matrix JSON and Figure 2.

Requires `SDG_UN_data.zip` in `<root>/sdggraph/` — see README section 2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
G = '/content/drive/MyDrive/sdg-llm-graph/sdggraph'
print('folder exists:', os.path.isdir(G))
for f in ['SDG_UN_data.zip', 'un_sdg_series_descriptions.json']:
    p = os.path.join(G, f)
    print(f, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else '')

In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/sdg-llm-graph/artifacts/un_sdg_series_descriptions.json',
            '/content/drive/MyDrive/sdg-llm-graph/sdggraph/un_sdg_series_descriptions.json')
print('copied')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'scipy', 'numpy', 'matplotlib',
                'seaborn', 'openpyxl'],
               check=True)
print('Dependencies ready.')

In [ ]:
import os, json, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats

warnings.filterwarnings('ignore')
print('Imports OK.')

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# All paths derive from one root: $SDG_ROOT, else config.yaml, else Drive on
# Colab, else ./sdg_data. See README section 1.
import os, sys

def _resolve_sdg_root():
    r = os.environ.get('SDG_ROOT')
    if r:
        return os.path.abspath(os.path.expanduser(r))
    for cand in ('config.yaml', os.path.join('..', 'config.yaml')):
        if os.path.exists(cand):
            with open(cand) as fh:
                for line in fh:
                    line = line.split('#')[0].strip()
                    if line.startswith('root:'):
                        v = line.split(':', 1)[1].strip().strip('"\'')
                        if v:
                            return os.path.abspath(os.path.expanduser(v))
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            return '/content/drive/MyDrive/sdg-llm-graph'
        except Exception:
            pass
    return os.path.abspath('./sdg_data')

SDG_ROOT   = _resolve_sdg_root()
GRAPH_DIR  = os.path.join(SDG_ROOT, 'sdggraph')
DRIVE_ROOT = os.path.join(SDG_ROOT, 'aurora_sdg_graph_full')
DATA_CACHE = os.path.join(DRIVE_ROOT, 'data_cache')
for _d in (SDG_ROOT, GRAPH_DIR, DRIVE_ROOT, DATA_CACHE):
    os.makedirs(_d, exist_ok=True)

print(f'SDG_ROOT   : {SDG_ROOT}')
print(f'GRAPH_DIR  : {GRAPH_DIR}')
print(f'DATA_CACHE : {DATA_CACHE}')

# The preprocessor writes all of its artefacts into GRAPH_DIR.
DRIVE_DIR = GRAPH_DIR
print(f'Artefact dir: {DRIVE_DIR}')


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Path to the zip of 17 UN SDG Excel files on Drive
ZIP_PATH     = os.path.join(DRIVE_DIR, 'SDG_UN_data.zip')

# Local extraction directory (scratch; safe to delete between runs)
EXTRACT_DIR  = os.path.join(SDG_ROOT, '_cache', 'sdg_excel')

# Cached CSV after first extraction (avoids re-reading Excel on re-runs)
CACHED_CSV   = os.path.join(SDG_ROOT, '_cache', 'sdg_combined.csv')

# Output file
OUTPUT_FILE  = os.path.join(DRIVE_DIR, 'sdg_interaction_matrix_v3_2.json')

# Pradhan et al. (2017) / Warchold et al. (2022) parameters
MIN_OBS_PER_COUNTRY = 3     # min time points per country per series pair
MIN_TOTAL_PAIRS     = 50    # min pooled observations for Spearman (v3: raised from 10)
ALPHA               = 0.05  # significance threshold

# v3 toggles — see new cells below
APPLY_DIRECTION_FIX     = True   # Pradhan §2: flip bad-when-high indicators
EXCLUDE_BOOLEAN_SERIES  = True   # Warchold §2.3: drop dummy indicators

# SDG labels
SDG_LABELS = [
    'No Poverty', 'Zero Hunger', 'Good Health and Well-being',
    'Quality Education', 'Gender Equality', 'Clean Water and Sanitation',
    'Affordable and Clean Energy', 'Decent Work and Economic Growth',
    'Industry Innovation and Infrastructure', 'Reduced Inequalities',
    'Sustainable Cities and Communities',
    'Responsible Consumption and Production', 'Climate Action',
    'Life Below Water', 'Life on Land',
    'Peace Justice and Strong Institutions', 'Partnerships for the Goals',
]
assert len(SDG_LABELS) == 17

os.makedirs(os.path.join(SDG_ROOT, '_cache'), exist_ok=True)

print('Configuration:')
print(f'  ZIP:                {ZIP_PATH}')
print(f'  ZIP exists:         {os.path.exists(ZIP_PATH)}')
print(f'  Cached CSV:         {CACHED_CSV}')
print(f'  CSV exists:         {os.path.exists(CACHED_CSV)}')
print(f'  Output:             {OUTPUT_FILE}')
print(f'  MIN_TOTAL_PAIRS:    {MIN_TOTAL_PAIRS}  (v3 raised from 10)')
print(f'  Direction fix:      {APPLY_DIRECTION_FIX}')
print(f'  Boolean exclusion:  {EXCLUDE_BOOLEAN_SERIES}')


---
## Step 1 — Load UN SDG Excel Files

Extracts the 17 Excel files from the zip, reads each one,
auto-detects the goal number from the filename, and concatenates
into a single long-format DataFrame.

On re-runs the cached CSV is loaded directly (~seconds).

In [ ]:
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Zip not found at {ZIP_PATH}\n'
        'Place the UN SDG Indicators archive (17 Excel files) at:\n'
        f'  {DRIVE_DIR}/SDG_UN_data.zip'
    )

if os.path.exists(CACHED_CSV):
    print(f'Loading cached CSV: {CACHED_CSV}')
    df_raw = pd.read_csv(CACHED_CSV, low_memory=False)
    print(f'  {len(df_raw):,} rows × {len(df_raw.columns)} columns')

else:
    # ── Extract zip ───────────────────────────────────────────────────────────
    print(f'Extracting {ZIP_PATH} ...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
        all_files = z.namelist()
    print(f'  {len(all_files)} files extracted')

    # ── Find Excel files ──────────────────────────────────────────────────────
    excel_files = sorted([
        os.path.join(EXTRACT_DIR, f)
        for f in os.listdir(EXTRACT_DIR)
        if f.endswith('.xlsx') or f.endswith('.xls')
    ])
    # Also search one level deep (zip may have a subfolder)
    if not excel_files:
        for root, dirs, files in os.walk(EXTRACT_DIR):
            for f in files:
                if f.endswith('.xlsx') or f.endswith('.xls'):
                    excel_files.append(os.path.join(root, f))
        excel_files = sorted(excel_files)

    print(f'  Found {len(excel_files)} Excel files:')
    for f in excel_files:
        print(f'    {os.path.basename(f)}')

    if not excel_files:
        raise RuntimeError('No Excel files found in zip. Check zip contents.')

    # ── Read each file and assign goal number ─────────────────────────────────
    chunks = []
    t0 = time.time()

    for fpath in excel_files:
        fname = os.path.basename(fpath)
        print(f'Reading {fname} ...', end=' ', flush=True)
        t_f = time.time()

        try:
            # Try to infer goal number from filename
            # Common patterns: Goal1.xlsx, SDG1.xlsx, goal_01.xlsx,
            # GOAL1_data.xlsx, 1.xlsx, SDG-1.xlsx etc.
            import re
            nums = re.findall(r'(?:^|[^0-9])(1[0-7]|[1-9])(?:[^0-9]|$)', fname)
            goal_num = int(nums[0]) if nums else None

            # Read the Excel file
            df = pd.read_excel(fpath, engine='openpyxl')
            print(f'{len(df):,} rows  cols={list(df.columns[:6])}  '
                  f'[{time.time()-t_f:.1f}s]')

            # Peek at first row to help with column detection
            if len(chunks) == 0:
                print(f'  Sample columns: {list(df.columns)}')
                print(f'  Sample row 0:   {df.iloc[0].to_dict()}')

            if goal_num:
                df['goal_int'] = goal_num
            chunks.append(df)

        except Exception as e:
            print(f'ERROR: {e}')

    # ── Concatenate all goals ─────────────────────────────────────────────────
    df_raw = pd.concat(chunks, ignore_index=True)
    df_raw.to_csv(CACHED_CSV, index=False)
    print(f'\nCombined: {len(df_raw):,} rows × {len(df_raw.columns)} columns')
    print(f'Saved cache: {CACHED_CSV}  ({time.time()-t0:.0f}s total)')

print(f'\nAll columns: {list(df_raw.columns)}')
print(f'\nSample row:')
print(df_raw.iloc[0])

---
## Step 2 — Detect Columns and Clean

The UN Excel files use specific column names. This cell auto-detects
the key columns (series code, country code, year, value, goal)
and builds the long-format DataFrame needed for Spearman computation.

**Run the cell above first** and check the printed column names.
If auto-detection fails, set the column names manually in the
`OVERRIDE_*` variables below.

In [ ]:
# ── Manual overrides — set these if auto-detection fails ─────────────────────
# Leave as None to use auto-detection
OVERRIDE_SERIES  = None   # e.g. 'SeriesCode'
OVERRIDE_GEO     = None   # e.g. 'GeoAreaCode'
OVERRIDE_TIME    = None   # e.g. 'TimePeriod'
OVERRIDE_VALUE   = None   # e.g. 'Value'
OVERRIDE_GOAL    = None   # e.g. 'Goal'   (or None if inferred from filename)

# ── Auto-detect columns ───────────────────────────────────────────────────────
cols_lower = {c.lower().strip().replace(' ', '').replace('_', ''): c
              for c in df_raw.columns}

candidates = {
    'series': ['seriescode', 'series', 'indicatorcode', 'indicator',
               'seriesid', 'code'],
    'geo':    ['geoareacode', 'countrycode', 'geocode', 'm49', 'iso3',
               'geoarea', 'country', 'area', 'nation'],
    'time':   ['timeperiod', 'year', 'time', 'period', 'date',
               'timeperiodstart', 'yearperiod'],
    'value':  ['value', 'obsvalue', 'observation', 'datavalue', 'val'],
    'goal':   ['goal', 'goalcode', 'sdg', 'goalnumber', 'goalid'],
}

col_map = {}
for canon, options in candidates.items():
    override = locals().get(f'OVERRIDE_{canon.upper()}')
    if override:
        col_map[canon] = override
        continue
    for opt in options:
        key = opt.replace(' ', '').replace('_', '')
        if key in cols_lower:
            col_map[canon] = cols_lower[key]
            break

print('Column mapping:')
for k, v in col_map.items():
    print(f'  {k:10s} -> "{v}"')

missing = [k for k in ['series', 'geo', 'time', 'value'] if k not in col_map]
if missing:
    print(f'\nWARNING: could not detect columns for: {missing}')
    print('Set OVERRIDE_* variables above and re-run this cell.')
else:
    print('\nAll required columns detected.')

# ── Rename and clean ──────────────────────────────────────────────────────────
df = df_raw.rename(columns={v: k for k, v in col_map.items()}).copy()

# Parse goal number
def parse_goal(g):
    try:
        s = str(g).strip().upper().replace('SDG','').replace('GOAL','')
        return int(float(s.strip()))
    except:
        return None

if 'goal' in df.columns:
    df['goal_int'] = df['goal'].apply(parse_goal)
elif 'goal_int' not in df.columns:
    print('WARNING: no goal column found — goal_int must be set from filenames')

df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value', 'goal_int', 'geo', 'time', 'series'])
df['goal_int'] = df['goal_int'].astype(int)
df = df[df['goal_int'].between(1, 17)]

# Deduplicate
df = (df.groupby(['series', 'goal_int', 'geo', 'time'], as_index=False)
       ['value'].mean())

print(f'\nAfter cleaning: {len(df):,} rows')
print(f'SDGs present:   {sorted(df["goal_int"].unique())}')
print(f'Series codes:   {df["series"].nunique():,}')
print(f'Countries:      {df["geo"].nunique():,}')
print(f'Time range:     {df["time"].min()} – {df["time"].max()}')

# Build series → SDG mapping
series_to_sdg = (df[['series', 'goal_int']].drop_duplicates()
                 .set_index('series')['goal_int'].to_dict())
sdg_series = {g: [] for g in range(1, 18)}
for s, g in series_to_sdg.items():
    sdg_series[g].append(s)

print('\nSeries count per SDG:')
for g in range(1, 18):
    n = len(sdg_series[g])
    bar = '█' * min(n, 30)
    print(f'  SDG {g:2d}: {n:3d}  {bar}')

In [ ]:
# ── Pivot to country-level time series ───────────────────────────────────────
print('Pivoting to (country, time) × series ...')
pivot = df.pivot_table(
    index=['geo', 'time'],
    columns='series',
    values='value',
    aggfunc='mean'
)
print(f'Pivot shape: {pivot.shape}')
print(f'Non-null:    {pivot.notna().sum().sum():,} values')

# Build per-country dataframes for efficient access
print('Grouping by country ...')
country_data = {}
for country in pivot.index.get_level_values('geo').unique():
    try:
        sub = pivot.loc[country]
        if not sub.empty:
            country_data[country] = sub
    except:
        pass
print(f'Countries with data: {len(country_data):,}')

In [ ]:
# ── v3: Exclude Boolean / dummy indicators (Warchold 2022 §2.3) ──────────────
# Drop any series whose non-null values take only 1 or 2 distinct values.
# These are typically yes/no policy indicators (e.g. "country has X law")
# which bias Spearman correlation toward extreme values.

if EXCLUDE_BOOLEAN_SERIES:
    n_unique = pivot.nunique(dropna=True)
    boolean_series = n_unique[n_unique <= 2].index.tolist()
    print(f'Boolean/dummy series detected: {len(boolean_series)}')
    if boolean_series:
        print('  Examples (first 10):')
        for s in boolean_series[:10]:
            n_obs = pivot[s].notna().sum()
            print(f'    {s}  ({n_obs} non-null obs, {pivot[s].nunique()} unique)')
    pivot = pivot.drop(columns=boolean_series)
    print(f'Pivot after Boolean exclusion: {pivot.shape}')
else:
    print('Boolean exclusion DISABLED — set EXCLUDE_BOOLEAN_SERIES=True to enable.')


In [ ]:
# ── v3.1: Fetch indicator descriptions from UN SDG API ──────────────────────
# The UN Excel exports do not include a description column for the series codes.
# We fetch descriptions from the official API:
#   https://unstats.un.org/sdgapi/v1/sdg/Series/List
# This is a one-time fetch (cached to Drive); without it, the direction
# classifier has nothing to keyword-match against and excludes >95% of series.

import requests, os, json

API_URL = 'https://unstats.un.org/sdgapi/v1/sdg/Series/List'
API_CACHE = os.path.join(DRIVE_DIR, 'un_sdg_series_descriptions.json')

if os.path.exists(API_CACHE):
    print(f'Loading cached descriptions from {API_CACHE}')
    with open(API_CACHE) as f:
        series_meta = json.load(f)
else:
    print(f'Fetching from {API_URL} ...')
    r = requests.get(API_URL, timeout=60)
    r.raise_for_status()
    series_meta = r.json()
    with open(API_CACHE, 'w') as f:
        json.dump(series_meta, f)
    print(f'Cached to {API_CACHE}')

print(f'Series metadata entries: {len(series_meta):,}')

# Build code -> description lookup
series_desc = {entry['code']: entry.get('description', '').strip()
               for entry in series_meta}

# Coverage check against the pivot's series codes
codes_in_pivot = set(pivot.columns)
covered = sum(1 for c in codes_in_pivot if series_desc.get(c, '').strip())
print(f'\nCoverage: {covered}/{len(codes_in_pivot)} pivot series have descriptions '
      f'({100*covered/max(len(codes_in_pivot),1):.1f}%)')

# Show a few examples
print('\nSample (first 5 pivot series with descriptions):')
shown = 0
for code in list(codes_in_pivot):
    if series_desc.get(code, '').strip():
        print(f'  {code}: {series_desc[code][:100]}')
        shown += 1
        if shown >= 5:
            break

# Show any pivot codes the API does NOT know about
missing = [c for c in codes_in_pivot if not series_desc.get(c, '').strip()]
if missing:
    print(f'\nNo description from API for {len(missing)} pivot codes:')
    for c in missing[:10]:
        print(f'  {c}')
    if len(missing) > 10:
        print(f'  ... and {len(missing) - 10} more (will fall back to code-only classification)')


In [ ]:
# ── v3: Direction-of-progress lookup (Pradhan 2017 §2) ───────────────────────
# For each indicator series, assign a direction of +1 (good-when-high) or -1
# (good-when-low) based on a keyword heuristic on the series description.
# Indicators that cannot be classified confidently are flagged and EXCLUDED
# from the analysis until the user assigns them manually via DIRECTION_OVERRIDES.

# Manual overrides — fill in after seeing the "UNCERTAIN" output below
# Format: { 'SeriesCode': +1 or -1, ... }
DIRECTION_OVERRIDES = {
    # Example:
    # 'SI_POV_DAY1': -1,        # poverty headcount — good-when-low
    # 'SE_TOT_PRYREAD': +1,     # reading proficiency — good-when-high
}

# Keywords that indicate "good-when-low" — indicator should be MINIMIZED
NEGATIVE_KEYWORDS = [
    # mortality / morbidity / disease
    'mortality', 'death', 'fatal', 'fatalit', 'casualt', 'killed', 'killings of',
    'incidence', 'prevalence of', 'morbidit', 'infection',
    'hiv', 'tuberculosis', 'malaria', 'hepatitis', 'cholera', 'epidemic',
    'pandemic', 'outbreak', 'cancer mortality',
    'conflict-related death',
    'anaemia', 'anemia',                                # 2.2.3 maternal nutrition
    'adolescent birth',                                  # 3.7 — wanted lower
    'requiring interventions',                           # 3.3.5 NTD
    # poverty / deprivation
    'poverty', 'poor population', 'deprivation', 'destitut', 'slum',
    'hungry', 'undernourish', 'undernutrition', 'food insecur',
    'stunt', 'wasting', 'wasted', 'malnut',
    'open defecation',                                   # 6.2
    # unemployment / informality / inequality
    'unemployment', 'not in employment', 'informal employment',
    'inequality', 'gini', 'wage gap', 'gender gap', 'pay gap',
    'out of school', 'out-of-school',
    'migrant recruitment cost', 'recruitment cost',     # 10.7 — high cost = bad
    # violence / crime / conflict / harm
    'violence', 'homicide', 'assault', 'abuse', 'trafficking',
    'crime', 'corruption', 'bribery', 'discrimination',
    'married or in a union before',                     # child marriage
    'detain',                                            # detainees in prison
    'alcohol consumption', 'tobacco use',                # substance use
    'female genital mutilation', 'fgm',                 # 5.3
    'subjected to',                                      # 16.1.3 violence/assault
    'experienced sexual violence',                       # 16.2
    'enforced disappearance',                            # 16.10
    'illegal, unreported', 'illegal',                    # 14.6 IUU fishing
    # environmental / emissions / loss
    'emission', 'co2', 'greenhouse', 'pollution', 'pollutant',
    'particulate', 'pm2.5', 'pm10', 'so2', 'nox',
    'deforest', 'forest loss', 'land degradation', 'desertification',
    'biodiversity loss', 'species threat', 'extinct', 'red list',
    'depletion', 'overfish', 'acidif', 'fish stock collapse',
    'waste generation', 'plastic waste',
    'hazardous waste',                                   # 12.4
    'waste generated',                                   # generic waste indicators
    'food loss', 'food waste', 'food price anomal',     # 12.3, 2.c
    'abnormally high',                                   # 2.c food price anomalies
    # disaster / displacement
    'disaster', 'displaced', 'refugee', 'fragility', 'fragile',
    'damage', 'loss due to',
    'damaged or destroyed', 'destroyed dwelling',        # 11.5
    # debt / illicit / subsidies (negative direction)
    'debt service', 'arrears', 'subsidy to fossil',
    'external debt', 'debt stock',                       # 17.4
    'illicit',                                           # 16.4
    'fossil fuel', 'fossil-fuel',                        # 12.c
    'export subsidies',                                  # 2.b
    # consumption that should decrease (resource intensity, footprint)
    'material footprint', 'resource intensity', 'energy intensity',
    'water stress', 'water scarcity',
]

# Keywords that strongly indicate "good-when-high" (used only to override
# false negatives from the keyword list above — e.g. "access to electricity"
# contains no negative keyword, defaults to +1 anyway, so this is mostly
# documentation of the implicit positive direction)
POSITIVE_HINTS = [
    # education / capability
    'access to', 'coverage', 'enrollment', 'enrolment', 'literacy',
    'completion rate', 'completion', 'attendance',
    'proficiency', 'achievement', 'graduates',
    'participation rate',
    # economic
    'gdp', 'income', 'wages', 'expenditure on',
    'tax revenue', 'government revenue',                 # 17.1
    'agriculture share',                                  # AG_XPD_AGSGB
    'manufacturing value added',                          # NV_IND_MANF*
    'productivity',                                       # PD_AGR_*
    # energy / environment
    'renewable', 'clean', 'efficient',
    'protected area', 'reserve', 'conservation', 'protected',
    'green', 'sustainable',
    # gender / inclusion
    'representation of women', 'women in parliament', 'women in management',
    'women in managerial', 'women in senior',
    'seats held by women', 'seats in national parliament',
    'informed decisions',
    # services / infrastructure / health
    'safely managed', 'improved water', 'improved sanitation', 'handwashing',
    'safely treated',                                     # EN_WWT_TREAT_SF
    'good ambient water quality',                         # EN_H2O_*AMBQ
    'skilled health personnel', 'birth attended',         # SH_STA_BRTC
    'vaccine', 'vaccinated', 'vaccination',               # SH_ACS_*
    'health worker',                                      # SH_MED_*
    # R&D / innovation
    'r&d', 'research and development', 'patents',
    'researchers',
    'medium and high-tech', 'high-tech',
    # finance / inclusion
    'official development assistance',
    'official development',                               # broader
    'official flows',                                     # DC_TOF_*
    'assistance for development',                         # DC_TRF_*
    'resource flows for development',                     # DC_TRF_TFDV
    'climate-specific financial', 'climate finance',      # DC_FIN_CLIM*
    'climate financing',
    'financial support',                                  # DC_FIN_TOT/CLIMB/etc
    'private finance', 'private grants',                  # DC_OSSD_*
    'mobilised private',                                  # DC_OSSD_MPF
    'contributions provided',                             # DC_FIN_GEN
    'bank account', 'account at a financial institution',
    'commercial bank branches',
    # social protection
    'social protection', 'social assistance', 'social insurance',
    'maternity cash benefit',                             # SI_COV_MATNL
    'child benefit', 'pension',                           # SI_COV_*
    # technology / environment
    'environmentally sound', 'environment sound',         # DC_ENVTECH_*
    'tracked exported', 'tracked imported',               # DC_ENVTECH_*
    # geographic extents (default: more is better)
    'forest area', 'land area', 'mountain area',
    'wetland', 'mangrove', 'mountain green',              # ecosystem extents
    # biodiversity / genetic resources
    'biomass',                                            # AG_LND_FRSTBIOPHA
    'genetic resources', 'local breeds', 'breeds kept',
    'plant genetic',
    'fish stocks within biologically sustainable',
    # recycling
    'recycled', 'recycling',                              # EN_*_RCY*
    # remittance volume (cost variants are caught by NEGATIVE 'remittance cost')
    'volume of remittance', 'remittance volume',          # BX_TRF_PWKR
    # statistical / governance capacity
    'national plans', 'national strategies',              # SG_DSR_*
    'adopt and implement',                                # SG_DSR_SILS
    'national statistical',                               # SG_STT_*
    'birth registration', 'death registration',           # SG_REG_*
    'legal frameworks that promote',                      # SG_LGL_GENEQ*
    'compliance with', 'in compliance',                   # SG_NHR_IMPL
    'satisfied with',                                     # SP_PSR_SATIS_*
    'standard accounting',                                # ST_EEV_*
    'tracked',                                            # DC_ENVTECH_*
    # connectivity
    'mobile telephone', 'mobile network', 'broadband',
    'using the internet',
]

import re

def classify_direction(series_code, description=''):
    """Return (+1, -1, 0). 0 means uncertain → exclude."""
    if series_code in DIRECTION_OVERRIDES:
        return DIRECTION_OVERRIDES[series_code]
    text = (str(description) + ' ' + str(series_code)).lower()
    # Strong negative-direction signal
    for kw in NEGATIVE_KEYWORDS:
        if kw in text:
            return -1
    # Weak positive signal — anything with a positive hint is +1
    for kw in POSITIVE_HINTS:
        if kw in text:
            return +1
    # Default: most modern UN SDG indicators are good-when-high after
    # the 2015 framework redesign (rates of access, coverage, etc.)
    # But we return 0 for "uncertain" so the user sees what got defaulted.
    return 0

# Description lookup comes from the UN SDG API (fetched in the previous cell).
# The variable `series_desc` maps series code -> description string.
desc_lookup = series_desc
print(f'Using UN SDG API descriptions for keyword matching ({len(desc_lookup):,} entries available).')

# Classify every series in the pivot
series_in_pivot = list(pivot.columns)
direction = {}
uncertain = []
for s in series_in_pivot:
    d = classify_direction(s, desc_lookup.get(s, ''))
    direction[s] = d
    if d == 0:
        uncertain.append(s)

n_pos = sum(1 for v in direction.values() if v == +1)
n_neg = sum(1 for v in direction.values() if v == -1)
n_unk = sum(1 for v in direction.values() if v == 0)
print(f'Direction classification: +1 = {n_pos}, -1 = {n_neg}, uncertain = {n_unk}')

# Show uncertain series so user can manually classify
if uncertain:
    print()
    print(f'UNCERTAIN ({len(uncertain)} series — will be EXCLUDED from analysis):')
    print('Copy the codes below into DIRECTION_OVERRIDES with +1 or -1 to include them.')
    print()
    for s in uncertain[:40]:
        desc = desc_lookup.get(s, '')[:90]
        print(f'    \'{s}\':  # {desc}')
    if len(uncertain) > 40:
        print(f'    ... and {len(uncertain) - 40} more (see `uncertain` list)')

# Apply direction signs to the pivot table (in place, only on classified series)
# Drop uncertain series from pivot so they are not included in Spearman
classified_series = [s for s in series_in_pivot if direction[s] != 0]
dropped = len(series_in_pivot) - len(classified_series)
print(f'\nApplying direction signs to {len(classified_series)} classified series.')
print(f'Excluding {dropped} uncertain series from the Spearman analysis.')

if APPLY_DIRECTION_FIX:
    pivot = pivot[classified_series].copy()
    for s in classified_series:
        if direction[s] == -1:
            pivot[s] = -pivot[s]   # flip bad-when-high so all series are
                                   # "higher = closer to the SDG target"
    print('Direction fix APPLIED: all series now point in the same direction')
    print('(positive Spearman → synergy, negative → trade-off, in the Pradhan sense).')
else:
    pivot = pivot[classified_series].copy()
    print('Direction fix DISABLED — pivot trimmed to classified series only,')
    print('but no sign-flipping is applied. Set APPLY_DIRECTION_FIX=True to enable.')

print(f'Pivot shape after direction step: {pivot.shape}')


In [ ]:
# ── v3.2: Export uncertain series for colleague review ──────────────────────
# Writes the list of indicators whose direction couldn't be auto-classified
# to a CSV file on Drive. Three columns: series code, description, and a
# blank "direction" column for the colleague to fill in (+1 / -1 / ?).
# After colleague review, paste their answers into DIRECTION_OVERRIDES at
# the top of the previous cell and re-run from there.
#
# Safe to re-run: overwrites the CSV with the latest uncertain list (which
# shrinks as DIRECTION_OVERRIDES grows).

import csv

UNCERTAIN_CSV = os.path.join(DRIVE_DIR, 'sdg_direction_uncertain_for_review.csv')

if not uncertain:
    print('No uncertain series — nothing to export. The keyword classifier')
    print('handled everything in the pivot. (Skipping CSV write.)')
else:
    with open(UNCERTAIN_CSV, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['series_code', 'description', 'direction'])
        for s in uncertain:
            w.writerow([s, desc_lookup.get(s, ''), ''])
    print(f'Wrote {len(uncertain)} uncertain series to:')
    print(f'  {UNCERTAIN_CSV}')
    print()
    print('Next step:')
    print('  1. Download this CSV from Drive.')
    print('  2. Send to sustainability colleague along with')
    print('     SDG_Direction_Review_for_Colleague.md (the instructions doc).')
    print('  3. When they return the filled-in CSV, paste their +1/-1 calls')
    print('     into DIRECTION_OVERRIDES at the top of the previous cell.')
    print('  4. Re-run from the previous cell onward; the matrix will update.')


---
## Step 3 — Compute Spearman Correlations

For each of the 136 SDG pairs, loops over all indicator combinations,
pools observations across countries, computes Spearman ρ,
and classifies as synergy or trade-off.

**Runtime: ~5–15 minutes** depending on how many series are present.

In [ ]:
# Run in a new cell while Spearman is running
print('Series per SDG:')
for g in range(1, 18):
    print(f'  SDG {g:2d}: {len(sdg_series[g]):3d} series')

# Estimate worst-case inner loop iterations
worst = 0
from itertools import combinations
for gi, gj in combinations(range(1, 18), 2):
    n = len(sdg_series[gi]) * len(sdg_series[gj]) * len(country_data)
    if n > worst:
        worst = n
        worst_pair = (gi, gj)
print(f'\nWorst pair: SDG{worst_pair[0]} × SDG{worst_pair[1]} = {worst:,} iterations')
print(f'Countries: {len(country_data)}')

In [ ]:
import torch
import numpy as np
from itertools import combinations
import time

N = 17
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

n_syn = np.zeros((N, N), dtype=int)
n_trd = np.zeros((N, N), dtype=int)
n_tot = np.zeros((N, N), dtype=int)

# v2 addition: per-country accumulators
# Maps country code → (n_syn, n_trd, n_tot) numpy arrays of shape (17, 17)
per_country_n_syn = {}
per_country_n_trd = {}
per_country_n_tot = {}

print('Series per SDG:')
for g in range(1, 18):
    print(f'  SDG {g:2d}: {len(sdg_series[g]):2d} series')

# ── GPU Spearman helper ───────────────────────────────────────────────────────
def spearman_gpu(x: torch.Tensor, y: torch.Tensor):
    def rank(t):
        tmp = t.argsort()
        r = torch.zeros_like(tmp, dtype=torch.float32)
        r[tmp] = torch.arange(1, len(t) + 1, dtype=torch.float32,
                               device=t.device)
        return r
    rx, ry = rank(x), rank(y)
    rx = rx - rx.mean()
    ry = ry - ry.mean()
    denom = rx.norm() * ry.norm()
    if denom == 0:
        return 0.0, 1.0
    rho = (rx * ry).sum() / denom
    n = len(x)
    t_stat = rho * torch.sqrt(
        torch.tensor(n - 2, dtype=torch.float32, device=x.device)
        / (1 - rho ** 2 + 1e-10)
    )
    from scipy.stats import t as t_dist
    pval = 2 * t_dist.sf(abs(t_stat.item()), df=n - 2)
    return rho.item(), pval

# ── Build GPU tensor ──────────────────────────────────────────────────────────
print('\nBuilding GPU tensor from pivot table ...')
t0 = time.time()
pivot_gpu = torch.tensor(pivot.values, dtype=torch.float32).to(DEVICE)
col_index = {col: i for i, col in enumerate(pivot.columns)}
print(f'  Pivot on GPU: {pivot_gpu.shape}  [{time.time()-t0:.1f}s]')

# ── Spearman computation ──────────────────────────────────────────────────────
sdg_pairs = list(combinations(range(1, 18), 2))
print(f'\nComputing {len(sdg_pairs)} SDG pairs on {DEVICE} ...')
print(f'Parameters: alpha={ALPHA}, min_obs/country={MIN_OBS_PER_COUNTRY}, '
      f'min_total={MIN_TOTAL_PAIRS}')
t0 = time.time()

for idx, (gi, gj) in enumerate(sdg_pairs):
    si_list = [s for s in sdg_series.get(gi, []) if s in col_index]
    sj_list = [s for s in sdg_series.get(gj, []) if s in col_index]
    if not si_list or not sj_list:
        continue

    i, j = gi - 1, gj - 1

    for s_i in si_list:
        for s_j in sj_list:
            ci = col_index[s_i]
            cj = col_index[s_j]

            col_i = pivot_gpu[:, ci]
            col_j = pivot_gpu[:, cj]
            valid  = (~torch.isnan(col_i)) & (~torch.isnan(col_j))

            if valid.sum().item() < MIN_TOTAL_PAIRS:
                continue

            x = col_i[valid]
            y = col_j[valid]

            n_tot[i, j] += 1
            n_tot[j, i] += 1

            try:
                rho, pval = spearman_gpu(x, y)
                if np.isfinite(rho) and pval < ALPHA:
                    if rho > 0:
                        n_syn[i, j] += 1; n_syn[j, i] += 1
                    else:
                        n_trd[i, j] += 1; n_trd[j, i] += 1
            except:
                pass

    if (idx + 1) % 15 == 0 or idx == len(sdg_pairs) - 1:
        elapsed = time.time() - t0
        eta     = elapsed / (idx + 1) * (len(sdg_pairs) - idx - 1)
        print(f'  [{idx+1:3d}/{len(sdg_pairs)}]  '
              f'elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'Total indicator pairs: {n_tot.sum()//2:,}')
print(f'Synergies: {n_syn.sum()//2:,}  Trade-offs: {n_trd.sum()//2:,}')

---
## Step 4 — Build W Matrix

In [ ]:
mask = n_tot > 0
W    = np.zeros((N, N))
W[mask] = (n_syn[mask] - n_trd[mask]) / n_tot[mask].astype(float)
W_signed = np.where(mask, np.sign(W).astype(int), 0)

print('17×17 SDG Interaction Matrix W')
print('Positive = synergy  |  Negative = trade-off  |  -- = no data')
print()
header = '       ' + ''.join(f'{g:5d}' for g in range(1, 18))
print(header)
for i in range(N):
    row = f'SDG{i+1:2d}  '
    for j in range(N):
        if i == j:            row += '   ** '
        elif n_tot[i,j] == 0: row += '   -- '
        else:                 row += f'{W[i,j]:+.2f} '
    print(row)

print(f'\nSDG pairs synergy dominant (W>0):   {(W_signed>0).sum()//2}')
print(f'SDG pairs trade-off dominant (W<0): {(W_signed<0).sum()//2}')
print(f'SDG pairs no data (W==0):           {((W_signed==0) & ~np.eye(N,dtype=bool)).sum()//2}')

---
## Step 5 — Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

W_plot = W.copy(); np.fill_diagonal(W_plot, np.nan)
sns.heatmap(
    W_plot, ax=axes[0],
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    xticklabels=[f'SDG{i+1}' for i in range(N)],
    yticklabels=[f'SDG{i+1}' for i in range(N)],
    linewidths=0.3, square=True
)
axes[0].set_title(
    'SDG Interaction Matrix W (v3.2)\n'
    '(Pradhan 2017 method + Warchold 2022 refinements)\n'
    'Green = synergy  |  Red = trade-off',
    fontsize=10
)

N_plot = n_tot.astype(float); np.fill_diagonal(N_plot, np.nan)
sns.heatmap(
    N_plot, ax=axes[1],
    cmap='Blues',
    annot=True, fmt='.0f', annot_kws={'size': 7},
    xticklabels=[f'SDG{i+1}' for i in range(N)],
    yticklabels=[f'SDG{i+1}' for i in range(N)],
    linewidths=0.3, square=True
)
axes[1].set_title(
    'Indicator Pair Coverage\n'
    '(valid pairs per SDG pair — higher = more reliable)',
    fontsize=10
)

plt.tight_layout()
fig_path = os.path.join(DRIVE_DIR, 'sdg_interaction_matrix_v3_2.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

---
## Step 6 — Save JSON to Drive

In [ ]:
country_matrices_W = {}
country_matrices_signed = {}
country_n_pairs = {}
for country in per_country_n_tot:
    cn_syn = per_country_n_syn.get(country, np.zeros((N, N), dtype=int))
    cn_trd = per_country_n_trd.get(country, np.zeros((N, N), dtype=int))
    cn_tot = per_country_n_tot[country]
    cW = np.where(cn_tot > 0, (cn_syn - cn_trd) / np.maximum(cn_tot, 1), 0.0)
    cW_signed = np.sign(cW)
    country_matrices_W[country] = cW.tolist()
    country_matrices_signed[country] = cW_signed.tolist()
    country_n_pairs[country] = int(cn_tot.sum() // 2)

print(f'Per-country matrices built: {len(country_matrices_W)} countries')
top_5 = sorted(country_n_pairs.items(), key=lambda x: -x[1])[:5]
print(f'Top 5 by pair count: {top_5}')

output = {
    'stats': {
        'source':              'UN Global SDG Indicators Database (17 Excel files)',
        'method':              'Spearman rank correlation, following Pradhan et al. (2017) with Warchold et al. (2022) refinements',
        'direction_fix':       APPLY_DIRECTION_FIX,
        'boolean_excluded':    EXCLUDE_BOOLEAN_SERIES,
        'alpha':               ALPHA,
        'min_obs_per_country': MIN_OBS_PER_COUNTRY,
        'min_total_pairs':     MIN_TOTAL_PAIRS,
        'n_countries':         len(country_data),
        'n_series':            len(series_to_sdg),
        'total_pairs_used':    int(n_tot.sum() // 2),
        'total_synergies':     int(n_syn.sum() // 2),
        'total_tradeoffs':     int(n_trd.sum() // 2),
        'citation_method': (
            'Pradhan, P., Costa, L., Rybski, D., Lucht, W., & Kropp, J. P. (2017). '
            'A systematic study of sustainable development goal (SDG) interactions. '
            "Earth's Future, 5(11), 1169-1179. "
            'https://doi.org/10.1002/2017EF000632'
        ),
        'citation_method_refinements': (
            'Warchold, A., Pradhan, P., Thapa, P., Putra, M. P. I. F., & Kropp, J. P. (2022). '
            'Building a unified sustainable development goal database: '
            'Why does sustainable development goal data selection matter? '
            'Sustainable Development, 30(5), 1278-1293. '
            'https://doi.org/10.1002/sd.2316'
        ),
        'citation_data': (
            'United Nations Statistics Division. '
            'Global SDG Indicators Database. '
            'https://unstats.un.org/sdgs/indicators/database/'
        ),
        'note': (
            'W[i,j] in [-1,+1]: positive = synergy dominant, '
            'negative = trade-off dominant, 0 = no data or neutral. '
            'matrix_signed encodes {-1, 0, +1}.'
        )
    },
    'sdg_labels':    SDG_LABELS,
    'matrix_W':      W.tolist(),
    'matrix_signed': W_signed.tolist(),
    'n_pairs':       n_tot.tolist(),
    'n_synergies':   n_syn.tolist(),
    'n_tradeoffs':   n_trd.tolist(),
    'country_matrices_W':      country_matrices_W,
    'country_matrices_signed': country_matrices_signed,
    'country_n_pairs':         country_n_pairs,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)

size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f'Saved: {OUTPUT_FILE}  ({size_kb:.0f} KB)')
print(f'\nDone! Upload complete to Drive.')

---
## Step 7 — Sanity Check

Known results from Pradhan (2017):
- SDG 1 (No Poverty) should show strong synergies with most goals
- SDG 12 (Responsible Consumption) should show trade-offs with several goals

In [ ]:
print('Sanity check vs Pradhan (2017):')
print()
print('SDG 1 (No Poverty) interactions:')
for j in range(N):
    if j == 0: continue
    label = '+SYN' if W_signed[0,j] > 0 else ('-TRD' if W_signed[0,j] < 0 else ' -- ')
    n     = n_tot[0, j]
    print(f'  SDG 1 <-> SDG{j+1:2d}  W={W[0,j]:+.3f}  {label}  (n={n})')

print()
print('SDG 12 (Responsible Consumption) interactions:')
for j in range(N):
    if j == 11: continue
    label = '+SYN' if W_signed[11,j] > 0 else ('-TRD' if W_signed[11,j] < 0 else ' -- ')
    n     = n_tot[11, j]
    print(f'  SDG12 <-> SDG{j+1:2d}  W={W[11,j]:+.3f}  {label}  (n={n})')

print()
print()
print('v3 expected qualitative patterns:')
print('  SDG 1  - mostly +SYN with SDG 3, 4, 5, 6, 7 (Pradhan 2017 strongest synergies)')
print('  SDG 12 - mostly -TRD with SDG 13, 14, 15 (consumption vs environment)')
print('  SDG 8  - row should have several -TRD with environmental SDGs (Pradhan 2017,')
print('           Warchold 2022 both report SDG 8 as one of the most trade-off-prone)')
print()
print('If SDG 1 and SDG 8 rows look mostly flat/positive, the direction lookup is')
print('still missing entries — check the UNCERTAIN list and add to DIRECTION_OVERRIDES.')

---
## Notes on Using the Matrix in SDG Experiments

```python
import json, numpy as np

with open(os.path.join(GRAPH_DIR, 'sdg_interaction_matrix_v3_2.json')) as f:
    sdg_data = json.load(f)

W_SDG        = np.array(sdg_data['matrix_W'])      # continuous [-1, +1]
W_SDG_signed = np.array(sdg_data['matrix_signed'])  # {-1, 0, +1}
n_pairs      = np.array(sdg_data['n_pairs'])         # coverage per cell

# For graph calibration: use W_SDG directly
# Mask sparse cells if needed:
W_SDG[n_pairs < 5] = 0
```

In [ ]:
import json, numpy as np
R = '/content/drive/MyDrive/sdg-llm-graph'
new = json.load(open(f'{R}/sdggraph/sdg_interaction_matrix_v3_2.json'))
ref = json.load(open(f'{R}/artifacts/sdg_interaction_matrix_v3_2.json'))
print(np.abs(np.array(new['matrix_W']) - np.array(ref['matrix_W'])).max(),
      new['n_pairs'] == ref['n_pairs'])